# Week 1 – ML Fundamentals + Data Preprocessing

## Mini Project: Titanic Survival Prediction – Data Cleaning Project

This notebook completes the Week 1 assignments: dataset exploration, missing-value handling, categorical encoding, train/test split, Linear Regression practice, and Titanic data cleaning/visualization.

**Input:** `Titanic-Dataset.csv` (Kaggle Titanic dataset) placed in the same folder as this notebook.

**Output:** `titanic_cleaned.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Load the Kaggle Titanic dataset
df = pd.read_csv('Titanic-Dataset.csv')
print('Dataset shape:', df.shape)
df.head(10)


## 1. Explore the dataset


In [ ]:
print('--- INFO ---')
df.info()

print('\n--- DESCRIPTIVE STATISTICS ---')
display(df.describe(include='all').T)

print('\n--- MISSING VALUES ---')
display(df.isnull().sum().sort_values(ascending=False).to_frame('missing_values'))


## 2. Handle missing data
Age is filled with the median because it is less sensitive to extreme values. Embarked is filled with its mode. Cabin has a very high percentage of missing values, so it is dropped.

In [ ]:
df_clean = df.copy()

# Drop Cabin because most values are missing
if 'Cabin' in df_clean.columns:
    df_clean.drop(columns=['Cabin'], inplace=True)

# Median imputation for Age
if 'Age' in df_clean.columns:
    df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Mode imputation for Embarked
if 'Embarked' in df_clean.columns:
    df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Fill Fare defensively if required
if 'Fare' in df_clean.columns:
    df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())

print('Remaining missing values:')
display(df_clean.isnull().sum().to_frame('missing_values'))


## 3. Encode categorical variables
For the required Titanic columns, `Sex` is label encoded and `Embarked` is one-hot encoded.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder for Sex
if 'Sex' in df_clean.columns:
    le = LabelEncoder()
    df_clean['Sex'] = le.fit_transform(df_clean['Sex'])
    print('Sex mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# One-hot encoding for Embarked
if 'Embarked' in df_clean.columns:
    df_clean = pd.get_dummies(df_clean, columns=['Embarked'], prefix='Embarked', dtype=int)

df_clean.head()


## 4. Remove non-useful identifier/text columns
PassengerId is only an identifier and Name/Ticket are not needed for this Week 1 cleaning deliverable.

In [ ]:
drop_cols = [c for c in ['PassengerId', 'Name', 'Ticket'] if c in df_clean.columns]
df_clean.drop(columns=drop_cols, inplace=True)

print('Columns after cleaning/encoding:')
print(df_clean.columns.tolist())


## 5. Visualize age distribution


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_clean['Age'], bins=30)
plt.title('Age Distribution of Titanic Passengers')
plt.xlabel('Age')
plt.ylabel('Number of Passengers')
plt.show()


## 6. Save the cleaned dataset


In [ ]:
df_clean.to_csv('titanic_cleaned.csv', index=False)
print('Saved: titanic_cleaned.csv')
print('Final shape:', df_clean.shape)


# Practice Set Solutions


In [ ]:
# 1. Load a CSV and print first 10 rows
practice_df = pd.read_csv('Titanic-Dataset.csv')
display(practice_df.head(10))


In [ ]:
# 2. Train/test split using sklearn
from sklearn.model_selection import train_test_split

# Example using Titanic features after preprocessing
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)


## 7. Linear Regression practice
The Titanic target `Survived` is binary, so Linear Regression is **not** the appropriate final model for survival classification. For the practice question, the following example demonstrates Linear Regression using a small house-price dataset.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Small practice dataset: area (sq ft) -> price
house = pd.DataFrame({
    'area': [800, 1000, 1200, 1500, 1800, 2000, 2200, 2500, 2800, 3000],
    'price': [80, 105, 125, 155, 185, 205, 225, 255, 285, 305]
})

Xh = house[['area']]
yh = house['price']
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(Xh_train, yh_train)
pred = model.predict(Xh_test)

print('R² score:', r2_score(yh_test, pred))
print('RMSE:', mean_squared_error(yh_test, pred) ** 0.5)
print('Coefficient:', model.coef_[0])
print('Intercept:', model.intercept_)


In [ ]:
# 8. Predict house price for a new area
new_area = 2000
predicted_price = model.predict(pd.DataFrame({'area': [new_area]}))[0]
print(f'Predicted price for {new_area} sq ft: {predicted_price:.2f} (in the same units as the training data)')


## Conclusion
- Dataset was loaded and explored using `info()`, `describe()`, and missing-value checks.
- Missing values were handled using median/mode imputation and a high-missingness column was removed.
- `Sex` was label encoded and `Embarked` was one-hot encoded.
- Age distribution was visualized.
- The cleaned Titanic dataset was exported as `titanic_cleaned.csv`.
- Train/test splitting and Linear Regression practice were also demonstrated.
